# 12장 학습 파이프라인 한 바퀴 — MNIST와 MLP

**2단계 · 주석 달기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_2_comment.ipynb)

이 노트북에는 **주석을 지운 코드**가 들어 있습니다. 아래 표의 「주석 달기」 과제대로 내 말로 주석을 달아 보세요. 주석을 다는 동안 코드를 한 줄씩 다시 읽게 됩니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 셀을 실행하면 내가 고친 내용이 파일에 저장됩니다.
- 그 아래 확인 셀을 실행해, 주석을 단 뒤에도 코드가 전과 똑같이 도는지 봅니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `train_mnist.py` 를 12.0의 주석 여섯 줄을 먼저 치고, 그 아래를 채우는 순서로 입력. 실행해 정확도 0.94 근처 확인 | [ch12_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_1_follow.ipynb) |
| 2. 주석 달기 | 반복문 안 다섯 줄에 ①~⑤를, `evaluate` 의 세 장치(`no_grad`·`eval`·`argmax`)에 이유를 주석으로 | [ch12_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_2_comment.ipynb) **(지금 이 노트북)** |
| 3. 다시 만들기 | 주석 여섯 줄만 남긴 빈 파일에서 전체를 복원. 그다음 `experiments.py` 의 `run` 함수처럼 감싸서 학습률 세 개를 비교 | [ch12_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "12"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/12")

## `train_mnist.py` — 대상: 파일 전체

In [ ]:
%%writefile train_mnist.py
import os
import time
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

config = {
    "data_root": os.environ.get("JUCOS_DATA", "./data"),
    "batch_size": 64,
    "hidden": 128,
    "lr": 0.1,
    "epochs": 2,
    "seed": 0,
    "log_every": 200,
}
torch.manual_seed(config["seed"])
device = torch.device("cpu")


transform = transforms.ToTensor()
train_set = datasets.MNIST(config["data_root"], train=True, download=True, transform=transform)
test_set = datasets.MNIST(config["data_root"], train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
test_loader = DataLoader(test_set, batch_size=1000)
print(f"학습 {len(train_set)}장 / 시험 {len(test_set)}장, 배치 {len(train_loader)}개")


class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        x = x.flatten(1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


model = MLP(784, config["hidden"], 10).to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=config["lr"])


def evaluate(model, loader):
    """시험 데이터 전체에 대한 정확도. 학습하지 않으므로 기울기를 끈다."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            pred = scores.argmax(dim=1)
            correct += (pred == labels).sum().item()
    model.train()
    return correct / len(loader.dataset)


print(f"학습 전 정확도: {evaluate(model, test_loader):.3f}")
start = time.time()
for epoch in range(1, config["epochs"] + 1):
    running = 0.0
    for step, (images, labels) in enumerate(train_loader, start=1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        scores = model(images)
        loss = loss_fn(scores, labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
        if step % config["log_every"] == 0:
            print(f"  에포크 {epoch} 걸음 {step:4d}/{len(train_loader)}  평균 손실 {running / config['log_every']:.4f}")
            running = 0.0
    acc = evaluate(model, test_loader)
    print(f"에포크 {epoch} 끝: 시험 정확도 {acc:.3f}  (경과 {time.time() - start:.0f}초)")

### 확인

In [ ]:
!python train_mnist.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- train_mnist.py